# 04 - GOLD: Construcción de features / Feature engineering

## SPA

**Propósito**
Partir de la tabla silver y construir las variables derivadas que alimentarán al modelo. Es la capa específica de este caso de uso, a diferencia de silver, que es de propósito general.

**Entrada**
`telco_churn.silver.customers_clean` (7043 filas x 21 columnas)

**Salida**
`telco_churn.gold.customer_features` (7043 filas x 28 columnas)

**Variables nuevas**

- `is_new_customer` — verdadero si el cliente lleva cero meses. Marca a los once que aún no tienen facturación.
- `tenure_max` — verdadero si lleva exactamente 72 meses, el valor que podría ser un tope de la ventana de datos.
- `n_support_services` — de 0 a 4, cuántos servicios de soporte tiene contratados.
- `n_entertainment_services` — de 0 a 2, cuántos servicios de entretenimiento tiene.
- `avg_historical_charge` — lo que ha pagado de media al mes desde que es cliente. Para los once de antigüedad cero se usa su cuota actual.
- `charge_ratio` — cuota actual dividida entre la media histórica. Por encima de 1 significa que le han subido el precio.
- `fiber_no_support` — verdadero si tiene fibra y ningún servicio de soporte. Recoge la interacción encontrada en la exploración, donde ese grupo se daba de baja al 60 % frente al 41,9 % del conjunto de la fibra.
- `automatic_payment` — verdadero si paga por domiciliación o tarjeta automática. Agrupa las cuatro formas de pago en los dos bloques que el análisis mostró separados.

**Decisiones de diseño**

Se descarta `total_charges` porque equivale casi por completo al producto de la antigüedad por la cuota mensual. La parte que sí aportaba queda recogida en `avg_historical_charge` y `charge_ratio`.

Se conservan `gender`, `phone_service` y `multiple_lines` pese a mostrar poca señal en el análisis univariante. Una variable que no separa por sí sola puede aportar en combinación con otras.

`customer_id` se mantiene en la tabla aunque no sea una variable predictiva, porque hace falta para identificar a quién corresponde cada predicción. Se excluye dentro del pipeline, no aquí.

Las variables categóricas siguen como texto. Su codificación ocurre dentro del pipeline del modelo para que el objeto que se despliegue sepa transformar datos crudos por sí mismo.

---

## ENG

**Purpose**
Start from the silver table and build the derived variables that will feed the model. This is the layer specific to this use case, unlike silver, which is general-purpose.

**Input**
`telco_churn.silver.customers_clean` (7,043 rows x 21 columns)

**Output**
`telco_churn.gold.customer_features` (7,043 rows x 28 columns)

**New variables**

- `is_new_customer` — true if the customer has zero months of tenure. Flags the eleven with no billing yet.
- `tenure_max` — true if tenure is exactly 72 months, the value that may be a cap in the data window.
- `n_support_services` — 0 to 4, how many support services the customer holds.
- `n_entertainment_services` — 0 to 2, how many entertainment services the customer holds.
- `avg_historical_charge` — average monthly amount paid since becoming a customer. For the eleven with zero tenure, their current fee is used.
- `charge_ratio` — current fee divided by the historical average. Above 1 means their price has gone up.
- `fiber_no_support` — true if the customer has fibre and no support services. Captures the interaction found during exploration, where that group churned at 60 % against 41.9 % for fibre overall.
- `automatic_payment` — true if the customer pays by direct debit or automatic card. Groups the four payment methods into the two blocks the analysis showed apart.

**Design decisions**

`total_charges` is dropped because it is almost entirely equivalent to tenure multiplied by monthly charges, and whatever it did contribute is now captured by `avg_historical_charge` and `charge_ratio`.

`gender`, `phone_service` and `multiple_lines` are kept despite showing little signal in the univariate analysis. A variable that does not separate on its own may still contribute in combination with others, and checking that during modelling is cheap.

`customer_id` stays in the table even though it is not a predictive variable, because it is needed to know which prediction belongs to whom. It is excluded inside the pipeline, not here.

Categorical variables remain as text. Their encoding happens inside the model pipeline so that the deployed object knows how to transform raw data on its own.

In [0]:
import pandas as pd
import numpy as np

from churn.config import (SILVER_TABLE, GOLD_TABLE, ID, TARGET)
from churn.features import (
    is_new_customer,
    tenure_max,
    n_support_services,
    n_entertainment_services,
    avg_historical_charge,
    charge_ratio,
    fiber_no_support,
    automatic_payment,
    )

from churn.validation import validar_silver


In [0]:
pdf = spark.table(SILVER_TABLE).toPandas()

In [0]:
validar_silver(pdf)

In [0]:
pdf["is_new_customer"] = is_new_customer(pdf)
pdf["tenure_max"] = tenure_max(pdf)
pdf["n_support_services"] = n_support_services(pdf)
pdf["n_entertainment_services"] = n_entertainment_services(pdf)
pdf["avg_historical_charge"] = avg_historical_charge(pdf)
pdf["fiber_no_support"]  = fiber_no_support(pdf)
pdf["automatic_payment"] = automatic_payment(pdf)
pdf["charge_ratio"] = charge_ratio(pdf["monthly_charges"], pdf["avg_historical_charge"])

In [0]:
# Comprobaciones

nuevos = pdf.loc[pdf["is_new_customer"], "charge_ratio"]

assert pdf["is_new_customer"].sum() == 11, f"is_new_customer: {pdf['is_new_customer'].sum()}"
assert pdf["tenure_max"].sum() == 362, f"tenure_max: {pdf['tenure_max'].sum()}"
assert pdf["n_support_services"].between(0, 4).all(), "n_support_services fuera de rango"
assert pdf["n_entertainment_services"].between(0, 2).all(), "n_entertainment_services fuera de rango"
assert np.isfinite(pdf["avg_historical_charge"]).all(), "avg_historical_charge tiene nulos o infinitos"
assert pdf["avg_historical_charge"].min() > 0, f"mínimo: {pdf['avg_historical_charge'].min()}"
assert np.isfinite(pdf["charge_ratio"]).all(), "charge_ratio tiene nulos o infinitos"
assert (nuevos == 1).all(), f"charge_ratio de clientes nuevos: {nuevos.unique()}"

print("Seis features validadas")

In [0]:
gold = pdf.drop(columns=["total_charges"])

print(f"{gold.shape[1]} columnas")
print(sorted(gold.columns))

In [0]:
from churn.validation import validar_gold

validar_gold(pdf)

In [0]:
spark.createDataFrame(gold).write.mode("overwrite").saveAsTable(GOLD_TABLE)

In [0]:
sdf = spark.table(GOLD_TABLE)

assert sdf.count() == 7043, f"Filas en gold: {sdf.count()}"
sdf.printSchema()

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {GOLD_TABLE}"))